# Documentation and Versioning

Adapted from the *Research Software Development* module by
[Kyle Niemeyer](https://kyleniemeyer.github.io/research-software-dev-modules/module-documentation/).

Examples throughout are drawn from the real [Cantera](https://github.com/Cantera/cantera) codebase — an open-source toolkit for chemical kinetics, thermodynamics, and transport.

## Why document?

> *"It works on my machine"* is not reproducible science.

Documentation is what turns code **you** can run into code **others** (and future-you) can run.
It costs little on top of the coding you already do, and pays back many times over:

* **Clarity** — colleagues can understand and build on your work
* **Provenance** — the scientific process behind a result is recorded
* **Competence** — well-documented software signals professional, trustworthy work

The good news: most documentation is a small, steady habit, not a separate project.

## Types of documentation

Documentation lives at several levels — from the whole project down to a single line:

| Level | Audience | Example |
|---|---|---|
| **README / guides** | Anyone arriving at the repo | `README.rst`, `INSTALL.md` |
| **API documentation** | People *calling* your code | docstrings → Sphinx site |
| **Self-documenting code** | People *reading* your code | clear names, structure |
| **Comments** | People *editing* a tricky section | `# why`, not `# what` |
| **Theory / user guides** | Domain users | tutorials, math derivations |

We'll walk up this ladder, then finish with **versioning** — how you label releases of all of it.

## 1. README files

The **README** sits at the top of the repository. It is the first — and often only — thing a newcomer reads. A good README answers, quickly:

* **What** is this? **Why** would I use it?
* **How** do I install it?
* **Where** do I go next (docs, examples, community)?

It is usually surrounded by companion files: `LICENSE`, `INSTALL`, `CITATION`, `CODE_OF_CONDUCT`, `CONTRIBUTING`, `CHANGELOG`.

### Example: Cantera's `README.rst`

Notice the structure — *what it is*, *what can it do*, then *install* and *where next*. The `|badges|` at the top show live status (CI, coverage, DOI, release) at a glance.

```rst
|doi| |codecov| |ci| |release|

What is Cantera?
================
Cantera is an open-source collection of object-oriented software tools for
problems involving chemical kinetics, thermodynamics, and transport processes.
Among other things, it can be used to:

* Evaluate thermodynamic and transport properties of mixtures
* Compute chemical equilibrium
* Conduct kinetics simulations with large reaction mechanisms
* Simulate one-dimensional flames

Installation
============
`Installation instructions for the current release of Cantera
<https://cantera.org/stable/install/index.html>`_ are available from the main site.

Documentation
=============
The `documentation <https://cantera.org>`_ offers a number of starting points:
- `Python tutorial <https://cantera.org/stable/userguide/python-tutorial.html>`_
- `Application Examples in Python <https://cantera.org/stable/examples/python/index.html>`_
```

*(abridged from `cantera/README.rst`)*

## 2. Code comments — comment the *why*, not the *what*

A comment that restates the code is noise; it just goes stale. Comment the things the code *can't* say: intent, units, the reason for a non-obvious choice.

### ❌ Too many comments

Every line is narrated. The comments add nothing the code doesn't already say — and now there are two things to keep in sync.

```python
def arrhenius(A, Ea, T):
    R = 8.314          # set the gas constant
    x = Ea / (R * T)   # divide Ea by R times T
    k = A * exp(-x)    # multiply A by the exponential of negative x
    return k           # return k
```

### ✅ Good names + a comment that earns its place

The names carry the meaning; the single comment records something the reader genuinely needs (units, and the source of the formula).

```python
R_GAS = 8.314  # J/(mol·K)

def arrhenius_rate(pre_exp, activation_energy, temperature):
    # Arrhenius form: k = A * exp(-Ea / R T)
    return pre_exp * exp(-activation_energy / (R_GAS * temperature))
```

## 3. Self-documenting code

The cheapest documentation is code that doesn't need any. Consistent **naming conventions** do most of the work. Python's [PEP 8](https://peps.python.org/pep-0008/) gives a widely-used set:

| Kind | Convention | Example |
|---|---|---|
| Packages / modules | `lowercase` | `cantera`, `composite` |
| Classes / Exceptions | `CamelCase` | `Solution`, `SolutionArray` |
| Functions / methods | `snake_case` | `restore_data`, `arrhenius_rate` |
| Constants | `ALL_CAPS` | `R_GAS` |
| Internal / private | `_leading_underscore` | `_import_pandas`, `_phase` |
| Magic methods | `__dunder__` | `__init__`, `__repr__` |

Guidelines: functions are **verbs** (`compute_equilibrium`), booleans read as questions (`is_ideal`, `has_transport`). Above all — **be consistent**.

### Cantera in practice

This snippet from `cantera/interfaces/cython/cantera/composite.py` follows the conventions without a single comment explaining *what* — `CamelCase` class, `_leading_underscore` for the deferred-import helper, `snake_case` function:

```python
class Solution(Transport, Kinetics, ThermoPhase):
    ...

_pandas = None
def _import_pandas():
    # defer import of pandas
    global _pandas
    ...
```

## 4. Docstrings

A **docstring** is a string literal right after a `def`/`class`/module statement, in triple quotes. Unlike a comment, it's part of the object at runtime — reachable via `help(obj)`, `obj?` in IPython, and `obj.__doc__`.

Try it on any standard-library function:

In [ ]:
help(print)


### Writing your own

A docstring should say what the function does, its **parameters**, and what it **returns**. Provide an example of how to use if it contains many parameters.

In [ ]:
def rescale(values, lo=0.0, hi=1.0):
    """Linearly rescale an array to the range [0, 1].

    Maps the minimum value of the input to 0 and the maximum to 1,
    scaling all other values proportionally in between.

    Parameters
    ----------
    input_array : numpy.ndarray
        The array of values to rescale.

    Returns
    -------
    numpy.ndarray
        An array the same shape as ``input_array``, with values
        linearly mapped onto the interval [0, 1].

    Examples
    --------
    >>> import numpy as np
    >>> rescale(np.array([1, 2, 3, 4, 5]))
    array([0.  , 0.25, 0.5 , 0.75, 1.  ])
    """
    ...

help(rescale)


### Docstring styles

Several conventions exist for the structured part. The two most common:

**NumPy style** — used in Cantera's `doc/sphinx/conf.py`:

```python
def executable_script(src_file, gallery_conf):
    """Validate if script has to be run according to gallery configuration.

    Parameters
    ----------
    src_file : str
        path to python script
    gallery_conf : dict
        Contains the configuration of Sphinx-Gallery

    Returns
    -------
    bool
        True if script has to be executed
    """
```

**reStructuredText (`:param:`) style** — used in Cantera's `composite.py`:

```python
def sort(self, col, reverse=False):
    """
    Sort SolutionArray by column ``col``.

    :param col: Column that is used to sort the SolutionArray.
    :param reverse: If True, the sorted list is reversed (descending order).
    """
```

Cantera uses both — the `napoleon` Sphinx extension understands NumPy/Google style, and Sphinx natively understands `:param:`. **Pick one style per project and stick to it.**

## 5. Generating a documentation website with Sphinx

[Sphinx](https://www.sphinx-doc.org) reads your docstrings (via `autodoc`) and hand-written pages and builds a browsable HTML site — this is how `https://cantera.org` is produced.

Getting started in a fresh project:

```bash
source .venv/bin/activate
pip install sphinx
sphinx-quickstart docs         # scaffold; choose 'separate source and build'
```

This creates `docs/` with a `conf.py` (configuration) and `index.rst` (the home page).

### Configuring `conf.py`

`conf.py` is where you enable extensions and wire up cross-references. A trimmed version of Cantera's:

```python
extensions = [
    'sphinx.ext.autodoc',       # pull in docstrings
    'sphinx.ext.autosummary',   # summary tables
    'sphinx.ext.intersphinx',   # link to other projects' docs
    'sphinx.ext.mathjax',       # render LaTeX math
    'myst_nb',                  # Markdown + Jupyter notebooks as pages
]

# napoleon lets autodoc understand NumPy / Google style docstrings
# (add 'sphinx.ext.napoleon' to the list above to enable it)

intersphinx_mapping = {
    'python': ('https://docs.python.org/3', None),
    'numpy':  ('https://numpy.org/doc/stable/', None),
    'pandas': ('https://pandas.pydata.org/pandas-docs/version/2.3', None),
}
```

`intersphinx` is the magic that turns a `numpy.ndarray` in your docstring into a live link to NumPy's own documentation. Installing `myst_parser`/`myst_nb` lets you author pages in Markdown (or even live notebooks like this one) instead of reStructuredText.

### Wiring up `index.rst` (the home page)

`conf.py` only *configures* Sphinx — it doesn't say what to document. That's the job of `index.rst`, the **landing page**. Out of the box `sphinx-quickstart` leaves it nearly empty, with just a `toctree` (the table of contents). Nothing references your code yet, so **no docstrings appear**.

The convention is to keep `index.rst` as a short, hand-written overview that *points* to other pages via its `toctree`, and put the auto-generated API on its own page (here, `api.rst`):

```rst
.. index.rst — the front door
summerschool-Jun2026-demo documentation
=======================================

A small demonstration package showing how docstrings and Sphinx fit together.

.. toctree::
   :maxdepth: 2
   :caption: Contents:

   api          ← links to api.rst
```

```rst
.. api.rst — the generated reference
API reference
=============

.. autosummary::

   rescale.rescale

.. automodule:: rescale.rescale
   :members:
```

`automodule` is what actually imports `rescale.rescale` and renders every docstring in it. The `toctree` in `index.rst` then stitches `index → api → rescale.rescale` into the site's navigation.

**Why a separate `api.rst`?** *Separation of concerns.* `index.rst` is narrative + a map; `api.rst` is generated reference. With one module you could inline `automodule` into `index.rst` — but the moment you add a second module, a tutorial, or examples, splitting them keeps the home page readable and lets each section grow independently. (For autodoc to find your package, it must be importable — e.g. `pip install -e .` — or add its path via `sys.path` in `conf.py`.)

### Building the site

```bash
sphinx-build -b html docs/source docs/build
# open docs/build/index.html
```

Cantera wraps this in its build system — `doc/SConscript` calls `sphinx-build -b html ...` to produce `build/doc/html/index.html`.

## 6. Publishing to GitHub Pages

A documentation site is most useful when it's **online and always current**. A GitHub Actions workflow can rebuild and redeploy on every push:

```yaml
# .github/workflows/sphinx.yml
on: push
jobs:
  docs:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.12' }
      - run: pip install sphinx myst-parser
      - run: sphinx-build -b html docs/source docs/build
      - uses: actions/upload-pages-artifact@v3
        with: { path: docs/build }
      - uses: actions/deploy-pages@v4
```

Then in **Settings → Pages**, set the source to GitHub Actions (or the `gh-pages` branch). Every push now keeps the public docs in sync with the code.

## 7. Versioning

Once people *depend* on your software, they need to know what changed between releases — and whether an update will break them. A **version number** communicates that. Common schemes:

* **SemVer** (Semantic Versioning) — meaning-carrying `MAJOR.MINOR.PATCH`
* **CalVer** — date-based, e.g. `2026.06`
* **ZeroVer** — perpetually `0.x` 😉

### Semantic Versioning: `MAJOR.MINOR.PATCH`

Given a version, you increment the:

* **MAJOR** when you make **incompatible** API changes
* **MINOR** when you add functionality in a **backward-compatible** way
* **PATCH** for backward-compatible **bug fixes**

Pre-`1.0.0` (`0.x.y`) means *anything may change* — public API not yet stable. Pre-releases get a suffix like `a1` (alpha), `b2` (beta), `rc1` (release candidate).

### Cantera's versions

Cantera demonstrates this exactly. From `SConstruct`:

```python
cantera_version = "4.0.0a1"   # 4.0.0, alpha 1 — a pre-release of the next MAJOR
```

and the current stable release recorded in `CITATION.cff`:

```yaml
version: "3.2.0"
date-released: 2025-11-17
doi: 10.5281/zenodo.17620923
```

So `3.2.0` is in the wild, while `4.0.0a1` is an **alpha** of a major release — the bump from `3.x` to `4.0` signals intentional, possibly breaking, API changes.

### Where the version lives

A common pattern: define the version **once**, derive everything else from it.

Some projects keep a `_version.py`:

```python
__version_info__ = (4, 0, 0, 'a1')
__version__ = '.'.join(map(str, __version_info__[:3]))
if len(__version_info__) == 4:
    __version__ += __version_info__[-1]      # -> '4.0.0a1'
```

and let the build read it (e.g. `pyproject.toml` with `dynamic = ["version"]`). Cantera instead sets it in `SConstruct` and threads it into the C++ headers, the Python module (`cantera.__version__`), **and** the docs (`conf.py` reads `CANTERA_VERSION` so the rendered site is always labelled with the right release). One source of truth, many consumers.

## Recap

Documentation is a ladder, and versioning labels each rung's releases:

1. **README** — the front door: what, why, how to install
2. **Comments** — explain the *why*, never the obvious *what*
3. **Self-documenting code** — consistent PEP 8 names do the heavy lifting
4. **Docstrings** — runtime-accessible API docs; pick one style (NumPy/Google/RST)
5. **Sphinx** — turns docstrings + pages into a website
6. **GitHub Pages + Actions** — keep that site live and current
7. **Versioning (SemVer)** — `MAJOR.MINOR.PATCH` tells users what an update means

None of it is a separate project — it's a habit layered onto the code you already write.